# 🏦 ML Workshop: From Fundamentals to a Kaggle Submission
## AIC Quant Meeting · Research Track · June 2026

**Estimated time: 60–90 minutes**

This notebook is your live coding companion for the AIC ML Workshop. It follows the workshop slides (11–20)
exactly — every concept on the slides is implemented and visualised here.

### What you'll build
```
Raw SEC fundamentals  →  Feature pipeline  →  Ridge baseline  →  LightGBM  →  Kaggle submission
```

### What you'll learn
- Why ML beats hand-crafted factor rules — and where it still struggles
- A 5-step feature engineering pipeline used by quantitative hedge funds
- How to evaluate predictive models using IC (the quant standard) and RMSE (the Kaggle metric)
- The #1 mistake in quant backtests — and exactly how to avoid it

### Timing guide
| Section | Slides | Time |
|---------|--------|------|
| Setup + Data exploration | 11–13 | 20 min |
| Feature engineering | 14–15 | 20 min |
| **Exercise 1** | 16 | 15 min |
| Ridge baseline | 17 | 10 min |
| **Exercise 2** | 17 | 10 min |
| Evaluation metrics | 18 | 10 min |
| **Exercise 3** | 18 | 5 min |
| LightGBM | 19 | 15 min |
| **Exercise 4** | 19 | 10 min |
| Pitfalls + submission | 20 | 10 min |

> **Tip:** Every section that starts with 🔧 is a hands-on exercise.
> Sections marked 📊 are visualisation cells — run them and study the output.
> Sections marked 🧠 are concept explanations — read the comments carefully.

---
## Section 1 — Setup *(2 min)*

Run this cell first. If you see import errors, run:
```bash
pip install scikit-learn lightgbm xgboost --break-system-packages
```

In [ ]:
# ── Install dependencies if missing ────────────────────────────────────────────────
import subprocess, sys

def ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        print(f"Installing {pkg}…")
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--break-system-packages",
                        "-q"], check=True)
        print(f"  ✓ {pkg} installed")

for pkg, imp in [("scikit-learn","sklearn"), ("lightgbm","lightgbm"), ("xgboost","xgboost")]:
    ensure(pkg, imp)

# ── Core imports ────────────────────────────────────────────────────────────────────
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr, pearsonr
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

# ── Paths ───────────────────────────────────────────────────────────────────────────
def find_repo_root() -> Path:
    """Find the repo root whether Jupyter starts in ./ or ./notebooks."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "train.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find data/raw/train.csv. Start Jupyter from the repo or notebooks folder.")

BASE       = find_repo_root()
TRAIN_PATH = BASE / "data" / "raw" / "train.csv"
TEST_PATH  = BASE / "data" / "raw" / "test.csv"
SUB_DIR    = BASE / "submissions"
SUB_DIR.mkdir(exist_ok=True)

TARGET = "return_pct"
ID_COL = "id"

print("✓ All imports OK")
print(f"  Train path: {TRAIN_PATH.resolve()}")
print(f"  Test  path: {TEST_PATH.resolve()}")

---
## Section 2 — Understanding the Data *(~15 min)*

### Slides 12–13: What are we predicting?

**The task:** predict `return_pct` — the percentage return of a US stock over the following 12 months.

This is a **cross-sectional** problem:
- We don't need to predict that Apple will return exactly 12.3%
- We only need to **rank stocks**: which ones will outperform?
- Even an IC of 0.03–0.08 is commercially valuable (industry benchmark)

**Why it's hard:**
- Signal is weak and very noisy
- A few extreme winners (+1000%) dominate any error metric
- Regime changes: factors that worked in 2020 may fail in 2022

### Key Reference: Fama-French (1993)
The academic foundation for fundamental investing:
- **Value premium**: cheap stocks (low P/B) outperform — ~0.26%/month historically
- **Size premium**: small caps outperform large caps
- **ML adds**: non-linear combinations the human researcher never thought to code

In [ ]:
# 📊 Load data and inspect

def read_competition_csv(path: Path) -> pd.DataFrame:
    """Read CSV and parse date columns only when the file contains them."""
    available_cols = pd.read_csv(path, nrows=0).columns
    date_cols = [c for c in ["period_start", "period_end"] if c in available_cols]
    return pd.read_csv(path, parse_dates=date_cols) if date_cols else pd.read_csv(path)

train = read_competition_csv(TRAIN_PATH)
test  = read_competition_csv(TEST_PATH)

print("═" * 55)
print(f"  Train shape: {train.shape[0]:,} rows × {train.shape[1]} cols")
print(f"  Test  shape: {test.shape[0]:,} rows × {test.shape[1]} cols")
print(f"  Train years: {sorted(train['start_year'].unique())}")
print(f"  Test  years: {sorted(test['start_year'].unique())}")
print("═" * 55)
print(f"\n  Target  : {TARGET}")
print(f"  Mean    : {train[TARGET].mean():.2f}%")
print(f"  Std     : {train[TARGET].std():.2f}%")
print(f"  Min     : {train[TARGET].min():.2f}%")
print(f"  Max     : {train[TARGET].max():.2f}%")
print(f"  25th pct: {train[TARGET].quantile(.25):.2f}%")
print(f"  75th pct: {train[TARGET].quantile(.75):.2f}%")
print("\n⚠️  Notice: the max return is enormous. This is the central challenge.")

In [ ]:
# 📊 Visualise the target distribution — the core challenge

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Distribution of return_pct (target variable)", fontsize=13, fontweight="bold")

# ── Panel 1: raw histogram (log scale needed) ──────────────────────────────────
ax = axes[0]
clipped = train[TARGET].clip(-200, 400)
ax.hist(clipped, bins=100, color="#2563EB", alpha=0.8, edgecolor="white")
ax.set_title("Raw distribution (clipped view)")
ax.set_xlabel("return_pct (%)")
ax.set_ylabel("Count")
ax.axvline(0, color="red", linewidth=1.5, label="0%")
ax.axvline(train[TARGET].mean(), color="orange", linewidth=1.5, label=f"Mean {train[TARGET].mean():.1f}%")
ax.legend()

# ── Panel 2: cumulative distribution ──────────────────────────────────────────
ax = axes[1]
sorted_ret = np.sort(train[TARGET].dropna())
pct = np.linspace(0, 100, len(sorted_ret))
ax.plot(sorted_ret.clip(-200, 500), pct, color="#2563EB")
ax.axvline(0, color="red", linewidth=1.5, linestyle="--")
ax.axhline(50, color="orange", linewidth=1, linestyle=":")
ax.set_title("Cumulative distribution (clipped)")
ax.set_xlabel("return_pct (%)")
ax.set_ylabel("Percentile")

# ── Panel 3: year-by-year median return ────────────────────────────────────────
ax = axes[2]
yr = train.groupby("start_year")[TARGET].agg(["median","mean"])
yr.plot(kind="bar", ax=ax, color=["#2563EB","#F97316"], edgecolor="white")
ax.set_title("Median & mean return by year")
ax.set_xlabel("start_year")
ax.set_ylabel("return_pct (%)")
ax.axhline(0, color="red", linewidth=1)
ax.legend(["Median","Mean"])
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

print("\n🧠 Key insight: The mean is pulled far above the median by extreme outliers.")
print("   This is the fat-tailed distribution that makes RMSE a problematic metric.")

In [ ]:
# 📊 Column overview — missingness matters as a signal!

print("Column overview:")
print("-" * 65)

cols_to_skip = [c for c in [ID_COL, "ticker", "start_year", "period_start", "period_end", TARGET]
                if c in train.columns]
feature_cols = [c for c in train.columns if c not in cols_to_skip]

summary = pd.DataFrame({
    "dtype":    train[feature_cols].dtypes,
    "non_null": train[feature_cols].count(),
    "missing%": (train[feature_cols].isnull().mean() * 100).round(1),
    "mean":     train[feature_cols].mean().round(2),
    "std":      train[feature_cols].std().round(2),
    "min":      train[feature_cols].min().round(2),
    "max":      train[feature_cols].max().round(2),
})
print(summary.to_string())

print(f"\n🧠 Columns with >50% missing:")
heavy = summary[summary["missing%"] > 50]["missing%"]
for col, pct in heavy.items():
    print(f"   {col}: {pct}% missing — missingness itself is a signal!")

In [ ]:
# 📊 Feature groups — understand what you're working with

RATIO_FEATS = [
    "pe_ttm", "price_to_book", "price_to_sales", "growth_pe_ratio",
    "gross_margin", "operating_margin", "net_margin",
    "roa", "roe", "rote",
    "revenue_growth_yoy", "revenue_growth_3y",
    "current_ratio", "quick_ratio", "debt_to_equity",
    "dividend_yield",
]

SCALE_FEATS = [
    "revenue_ttm", "net_income_ttm", "income_before_tax",
    "eps_basic", "eps_diluted",
    "total_assets", "stockholders_equity",
    "current_assets", "current_liabilities",
    "long_term_debt", "goodwill", "inventory",
    "dividends_ttm", "dividends_paid_ttm",
    "shares_outstanding", "shares_diluted",
]

SECTOR_COL = "sector_code"

print(f"Ratio features ({len(RATIO_FEATS)}):  will use signed_log + sector z-score")
print(f"  {RATIO_FEATS}")
print()
print(f"Scale features ({len(SCALE_FEATS)}):  will use log1p (always positive)")
print(f"  {SCALE_FEATS}")
print()
print(f"Sector column: {SECTOR_COL} — values: {sorted(train[SECTOR_COL].dropna().unique().astype(int))}")
print()
print("🧠 Sector code maps (approximate):")
sector_map = {0:"Energy", 1:"Materials", 2:"Industrials", 3:"Consumer Disc.",
              4:"Consumer Stpl.", 5:"Healthcare", 6:"Financials",
              7:"Info Tech", 8:"Comm. Services", 9:"Utilities", 10:"Real Estate"}
for k,v in sector_map.items():
    n = (train[SECTOR_COL]==k).sum()
    print(f"   {k}: {v:<20} ({n:,} rows)")

---
## Section 3 — Train / Validation / Test Split *(3 min)*

### Slide 12: The cardinal rule of financial ML

**Never randomly shuffle financial time-series data.**

sklearn's default `train_test_split` will let your 2022 data inform your 2019 model.
That's data leakage — your backtest will look great and your live model will fail.

```
Train (2019–2021)  │  Val (2022)  │  2023 gap  │  Test (2024 — Kaggle)
─────────────────────────────────────────────────────────────────────────
  fit model here   │  tune here   │  held out   │  never touch (submit)
```

In [ ]:
# Train / Validation split by year — never shuffle!

train_mask = train["start_year"] < 2022
valid_mask = ~train_mask

train_fold = train[train_mask].copy()
valid_fold = train[valid_mask].copy()

print(f"Train fold: {len(train_fold):,} rows | years {sorted(train_fold['start_year'].unique())}")
print(f"Valid fold: {len(valid_fold):,} rows | years {sorted(valid_fold['start_year'].unique())}")
print()

# ── Why NOT random split? ──────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split as tts
tr_rnd, va_rnd = tts(train, test_size=0.2, random_state=42)

print("⚠️  WRONG — random split would mix years:")
print(f"   'Train': contains years {sorted(tr_rnd['start_year'].unique())}")
print(f"   'Valid': contains years {sorted(va_rnd['start_year'].unique())}")
print()
print("✓  CORRECT — temporal split:")
print(f"   Train fold: years {sorted(train_fold['start_year'].unique())}")
print(f"   Valid fold: years {sorted(valid_fold['start_year'].unique())}")

---
## Section 4 — Feature Engineering Concepts *(~10 min reading + 15 min Exercise 1)*

### Slides 14–15: Why raw fundamentals fail

Three problems with raw P/E ratios (and most fundamentals):

1. **Extreme outliers**: A company losing money has negative P/E → your model gets -500 to +12,000
2. **Log-normal distribution**: Most financial ratios are right-skewed — log transforms help
3. **Sector incomparability**: P/E 30 is *cheap* for tech, *expensive* for utilities

### The 5-step pipeline (used by quantitative hedge funds)

```
Step 1: Winsorise   → clip pe_ttm to [1st pct, 99th pct] per column
Step 2: Signed-log  → sign(x) × log1p(|x|)  handles negative values + ratios
Step 3: Log1p       → log1p(x) for scale features (always ≥ 0)
Step 4: Sector z    → (x - sector_median) / sector_std  per sector_code
Step 5: Miss flags  → 1 if was NaN else 0  (absence = signal)
```

**Critical rule:** fit ALL statistics (percentiles, medians, stds) on the **train fold only**,
then apply identically to validation and test. Otherwise you leak.

In [ ]:
# 🧠 Demonstrate WHY transforms are necessary — before/after

cols_to_show = ["pe_ttm", "price_to_book", "roe", "gross_margin", "debt_to_equity", "revenue_ttm"]

fig, axes = plt.subplots(2, len(cols_to_show), figsize=(18, 6), squeeze=False)
fig.suptitle("Feature Engineering: Before vs After", fontsize=13, fontweight="bold")

for i, col in enumerate(cols_to_show):
    ax_before = axes[0][i]
    ax_after  = axes[1][i]

    raw = train_fold[col].dropna()

    # clip to sensible view range for before plot
    p1, p99 = raw.quantile(0.01), raw.quantile(0.99)
    raw_clipped = raw.clip(p1, p99)

    # signed_log transform
    transformed = np.sign(raw_clipped) * np.log1p(np.abs(raw_clipped))

    ax_before.hist(raw_clipped, bins=60, color="#EF4444", alpha=0.8, edgecolor="white")
    ax_before.set_title(f"{col}\n(raw)", fontsize=9)
    ax_before.set_ylabel("Count" if i == 0 else "")

    ax_after.hist(transformed, bins=60, color="#22C55E", alpha=0.8, edgecolor="white")
    ax_after.set_title(f"{col}\n(signed_log)", fontsize=9)
    ax_after.set_ylabel("Count" if i == 0 else "")

axes[0][0].set_ylabel("RAW → Count", fontweight="bold")
axes[1][0].set_ylabel("TRANSFORMED → Count", fontweight="bold")
plt.tight_layout()
plt.show()

print("🧠 Notice: raw pe_ttm is extremely right-skewed. After signed_log, it's near-symmetric.")
print("   This makes linear models (Ridge) work MUCH better.")

In [ ]:
# 🧠 Demonstrate sector neutralisation

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Sector Neutralisation: Why a P/E of 30 means different things", fontsize=12)

# ── Before: P/E by sector ──────────────────────────────────────────────────────
ax = axes[0]
sector_map = {0:"Energy", 1:"Materials", 2:"Industrials", 3:"Cons.Disc",
              4:"Cons.Stpl", 5:"Healthcare", 6:"Financials",
              7:"Info Tech", 8:"Comm.Svc", 9:"Utilities", 10:"Real Est"}

pe_data = (train_fold[["pe_ttm", SECTOR_COL]].dropna()
           .copy())
pe_data["pe_ttm"] = pe_data["pe_ttm"].clip(-50, 200)

grouped = []
labels  = []
for s in sorted(pe_data[SECTOR_COL].unique()):
    vals = pe_data[pe_data[SECTOR_COL]==s]["pe_ttm"]
    if len(vals) > 20:
        grouped.append(vals.values)
        labels.append(f"{sector_map.get(int(s), str(s))}")

ax.boxplot(grouped, labels=labels, vert=False, patch_artist=True,
           boxprops=dict(facecolor="#BFDBFE", alpha=0.7))
ax.set_title("P/E ratio by sector (raw)")
ax.set_xlabel("pe_ttm (clipped to [-50, 200])")
ax.tick_params(axis="y", labelsize=8)
ax.axvline(30, color="red", linewidth=1.5, linestyle="--", label="P/E=30")
ax.legend()

# ── After: z-score within sector ──────────────────────────────────────────────
ax = axes[1]
pe_data["sector_median"] = pe_data.groupby(SECTOR_COL)["pe_ttm"].transform("median")
pe_data["sector_std"]    = pe_data.groupby(SECTOR_COL)["pe_ttm"].transform("std")
pe_data["pe_zscore"]     = ((pe_data["pe_ttm"] - pe_data["sector_median"])
                             / pe_data["sector_std"].clip(lower=1e-6))

grouped_z = []
for s in sorted(pe_data[SECTOR_COL].unique()):
    vals = pe_data[pe_data[SECTOR_COL]==s]["pe_zscore"]
    if len(vals) > 20:
        grouped_z.append(vals.clip(-4, 4).values)

ax.boxplot(grouped_z, labels=labels, vert=False, patch_artist=True,
           boxprops=dict(facecolor="#BBF7D0", alpha=0.7))
ax.set_title("P/E z-score after sector neutralisation")
ax.set_xlabel("z-score (sector-neutral pe_ttm)")
ax.tick_params(axis="y", labelsize=8)
ax.axvline(0, color="red", linewidth=1.5, linestyle="--")

plt.tight_layout()
plt.show()

print("🧠 After z-scoring, every sector has the same centre and spread.")
print("   Now a z-score of -1 means 'cheap relative to sector peers' — comparable across sectors.")

---
## 🔧 Exercise 1 — Build the Feature Pipeline *(15 min)*

Implement the 5-step pipeline. The helper functions are provided — your job is to understand them,
run them, and inspect the outputs.

**What you'll produce:**
- `X_train_raw`, `X_valid_raw`, `X_test_raw` — feature matrices with transforms + missingness flags
- Remaining NaNs are handled inside the model pipelines with `SimpleImputer` or LightGBM's native missing-value logic
- `y_train`, `y_valid` — target vectors

In [ ]:
# ── Column definitions ─────────────────────────────────────────────────────────

RATIO_FEATS = [
    "pe_ttm", "price_to_book", "price_to_sales", "growth_pe_ratio",
    "gross_margin", "operating_margin", "net_margin",
    "roa", "roe", "rote",
    "revenue_growth_yoy", "revenue_growth_3y",
    "current_ratio", "quick_ratio", "debt_to_equity",
    "dividend_yield",
]
SCALE_FEATS = [
    "revenue_ttm", "net_income_ttm", "income_before_tax",
    "eps_basic", "eps_diluted",
    "total_assets", "stockholders_equity",
    "current_assets", "current_liabilities",
    "long_term_debt", "goodwill", "inventory",
    "dividends_ttm", "dividends_paid_ttm",
    "shares_outstanding", "shares_diluted",
]
SECTOR_COL  = "sector_code"
ALL_FEATS   = RATIO_FEATS + SCALE_FEATS + [SECTOR_COL]

print(f"Ratio features : {len(RATIO_FEATS)}")
print(f"Scale features : {len(SCALE_FEATS)}")
print(f"Total input cols: {len(ALL_FEATS)}")

In [ ]:
# ── Step-by-step pipeline functions ────────────────────────────────────────────

def signed_log(x: pd.Series) -> pd.Series:
    """Step 2: handle negative ratios — sign(x) * log1p(|x|)"""
    return np.sign(x) * np.log1p(np.abs(x))

def log1p_scale(x: pd.Series) -> pd.Series:
    """Step 3: for non-negative scale features — log(1 + x)"""
    return np.log1p(x.clip(lower=0))

def fit_winsorize(series: pd.Series, lo=0.01, hi=0.99):
    """Step 1 (fit): compute clip bounds on TRAIN only"""
    return series.quantile(lo), series.quantile(hi)

def apply_winsorize(series: pd.Series, lo_val, hi_val) -> pd.Series:
    """Step 1 (apply): clip using bounds computed on train"""
    return series.clip(lo_val, hi_val)

def fit_sector_stats(df: pd.DataFrame, col: str):
    """Step 4 (fit): compute sector median and std on TRAIN only"""
    med = df.groupby(SECTOR_COL)[col].transform("median")
    std = df.groupby(SECTOR_COL)[col].transform("std").clip(lower=1e-6)
    return med.values, std.values

def apply_sector_zscore(series: pd.Series, df: pd.DataFrame, med_map: dict, std_map: dict):
    """Step 4 (apply): subtract sector median from TRAIN, divide by TRAIN sector std"""
    sectors = df[SECTOR_COL].fillna(-1).astype(int)
    med = sectors.map(med_map).fillna(series.mean())
    std = sectors.map(std_map).fillna(1.0)
    return (series - med) / std

print("Helper functions defined.")
print("Next cell: run the full pipeline.")

In [ ]:
# ── Full feature engineering pipeline ─────────────────────────────────────────
# This is the reference implementation — study each step!

def build_features(df: pd.DataFrame,
                   clip_stats: dict = None,
                   sector_stats: dict = None,
                   fit: bool = False):
    """
    Build the feature matrix for a dataframe.

    Parameters
    ----------
    df          : input dataframe (train_fold, valid_fold, or test)
    clip_stats  : dict of {col: (lo_val, hi_val)} — computed on train, reused on val/test
    sector_stats: dict of {col: (med_map, std_map)} — computed on train, reused on val/test
    fit         : if True, compute clip_stats and sector_stats from df (train fold only)
    """
    if fit:
        clip_stats   = {}
        sector_stats = {}

    feats  = pd.DataFrame(index=df.index)
    miss_cols = []

    # ── Step 1+2: winsorise + signed_log for ratio features ────────────────────
    for col in RATIO_FEATS:
        x = df[col].copy()

        # missingness flag (Step 5) — must do BEFORE imputation
        flag_col = f"{col}_miss"
        feats[flag_col] = x.isnull().astype(float)
        miss_cols.append(flag_col)

        # Step 1: winsorise
        if fit:
            lo_val, hi_val = fit_winsorize(x.dropna())
            clip_stats[col] = (lo_val, hi_val)
        lo_val, hi_val = clip_stats[col]
        x = apply_winsorize(x, lo_val, hi_val)

        # Fill NaN with 0 before log (will be handled by imputer later)
        x = x.fillna(0)

        # Step 2: signed log
        x = signed_log(x)

        # Step 4: sector z-score
        if fit:
            tmp = pd.DataFrame({col: x, SECTOR_COL: df[SECTOR_COL]})
            med_vals = tmp.groupby(SECTOR_COL)[col].median()
            std_vals = tmp.groupby(SECTOR_COL)[col].std().clip(lower=1e-6)
            med_map  = med_vals.to_dict()
            std_map  = std_vals.to_dict()
            sector_stats[col] = (med_map, std_map)
        med_map, std_map = sector_stats[col]
        x = apply_sector_zscore(x, df, med_map, std_map)

        feats[col] = x

    # ── Step 3: log1p for scale features ───────────────────────────────────────
    for col in SCALE_FEATS:
        x = df[col].copy()

        flag_col = f"{col}_miss"
        feats[flag_col] = x.isnull().astype(float)
        miss_cols.append(flag_col)

        # absolute value — some eps_diluted can be negative
        x = log1p_scale(x.abs())
        feats[col] = x

    # ── Sector code as numeric feature ─────────────────────────────────────────
    feats[SECTOR_COL] = df[SECTOR_COL].fillna(-1)

    # ── Final imputation (SimpleImputer will handle remaining NaN) ─────────────
    return feats, clip_stats, sector_stats


# ── Build train and validation feature matrices ─────────────────────────────────
print("Building features…")
X_train_raw, clip_stats, sector_stats = build_features(
    train_fold, fit=True   # ← fit=True: compute stats from training data only
)
X_valid_raw, _, _ = build_features(
    valid_fold, clip_stats=clip_stats, sector_stats=sector_stats, fit=False
)
X_test_raw, _, _ = build_features(
    test, clip_stats=clip_stats, sector_stats=sector_stats, fit=False
)

y_train = train_fold[TARGET].values
y_valid = valid_fold[TARGET].values

print(f"✓  X_train: {X_train_raw.shape}  — {X_train_raw.isnull().sum().sum()} remaining NaNs")
print(f"✓  X_valid: {X_valid_raw.shape}")
print(f"✓  X_test:  {X_test_raw.shape}")
print(f"✓  y_train: mean={y_train.mean():.2f}%  std={y_train.std():.2f}%")
print(f"\nFeature groups:")
miss_f = [c for c in X_train_raw.columns if c.endswith("_miss")]
ratio_f = [c for c in X_train_raw.columns if c in RATIO_FEATS]
scale_f = [c for c in X_train_raw.columns if c in SCALE_FEATS]
print(f"  Ratio features (signed_log + sector-z): {len(ratio_f)}")
print(f"  Scale features (log1p):                 {len(scale_f)}")
print(f"  Missingness flags:                       {len(miss_f)}")
print(f"  Sector code:                             1")
print(f"  Total:                                   {X_train_raw.shape[1]}")

In [ ]:
# 📊 Exercise 1 CHECK — verify your transforms worked

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Exercise 1 Result: Feature distributions after transformation", fontsize=12, fontweight="bold")

check_cols = ["pe_ttm", "price_to_book", "gross_margin", "roe",
              "revenue_ttm", "total_assets", "debt_to_equity", "current_ratio"]

for i, col in enumerate(check_cols):
    ax = axes[i // 4][i % 4]
    vals = X_train_raw[col].dropna()
    vals = vals.clip(vals.quantile(0.01), vals.quantile(0.99))  # clip outliers for display
    ax.hist(vals, bins=50, color="#6366F1", alpha=0.85, edgecolor="white")
    ax.set_title(col, fontsize=9)
    ax.set_ylabel("Count" if i % 4 == 0 else "")
    ax.axvline(vals.mean(), color="red", linewidth=1.5, linestyle="--")

plt.tight_layout()
plt.show()

print("✓ All distributions should look roughly bell-shaped (or at least symmetric).")
print("  If any are still heavily skewed, we may need additional treatment.")
print()

# 🔧 YOUR TURN: which feature still looks skewed after transformation?
# Try plotting 'dividend_yield' before and after — why might it still be skewed?
# Hint: 93% of dividend_yield values are NaN (companies that don't pay dividends)
print("🔧 YOUR TURN: run the cell below to explore dividend_yield's missingness pattern")

In [ ]:
# 🔧 YOUR TURN — explore a problematic feature

col = "dividend_yield"
miss_pct = train_fold[col].isnull().mean() * 100

print(f"dividend_yield missingness: {miss_pct:.1f}%")
print()

# What kinds of companies pay dividends?
paid   = train_fold[train_fold[col] > 0][TARGET].describe()
no_pay = train_fold[train_fold[col].isnull()][TARGET].describe()

print("Return stats for DIVIDEND-PAYING stocks:")
print(paid.round(2).to_string())
print()
print("Return stats for NON-DIVIDEND stocks (NaN dividend_yield):")
print(no_pay.round(2).to_string())
print()
print("🧠 Insight: the missingness flag (dividend_yield_miss) is itself a signal!")
print("   Non-payers may systematically differ in returns. Our pipeline captures this.")

---
## Section 5 — Ridge Regression Baseline *(~10 min)*

### Slide 17: Three lines of code — one big lesson

Ridge regression is linear regression with an L2 penalty: it shrinks large coefficients toward zero.

$$\hat{y} = X\beta \quad \text{subject to} \quad \sum_j \beta_j^2 \leq t$$

**Why start here?**
- Interpretable: each feature has one coefficient — sign tells you direction
- Fast: fits on 18k rows in milliseconds
- Diagnostic: if Ridge can't find signal, complex models likely won't either
- Sanity check: low P/E should have positive coefficient (value premium)

In [ ]:
# ── Metric helpers ─────────────────────────────────────────────────────────────

def rmse(y_true, y_pred):
    """Root Mean Squared Error — the Kaggle metric (lower = better)"""
    return np.sqrt(mean_squared_error(y_true, y_pred))

def ic(y_true, y_pred):
    """Information Coefficient — Spearman rank correlation (higher = better)
    
    This is the quant finance standard metric.
    IC > 0 means you rank stocks in the correct direction.
    IC of 0.03–0.08 is considered good for fundamental models.
    """
    rho, pval = spearmanr(y_pred, y_true)
    return rho, pval

def naive_rmse(y_valid, y_train):
    """Baseline: always predict the training set mean"""
    naive_pred = np.full(len(y_valid), y_train.mean())
    return rmse(y_valid, naive_pred)

# ── Compute naive baseline ──────────────────────────────────────────────────────
baseline = naive_rmse(y_valid, y_train)
print(f"Naive baseline RMSE (always predict mean): {baseline:.2f}")
print(f"  → Any model must beat RMSE < {baseline:.2f} to add value")
print()
print("🧠 Why is the naive RMSE so large?")
print(f"   Training mean: {y_train.mean():.2f}%")
print(f"   But some stocks returned +5000%, +10000%…")
print(f"   RMSE squares those errors: (5000 - 12)² = enormous penalty!")

In [ ]:
# ── Build Ridge pipeline ───────────────────────────────────────────────────────

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Impute remaining NaNs with column median
# 2. Standardise (Ridge is sensitive to scale)
# 3. Ridge regression with L2 penalty

ridge_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge(alpha=1.0)),
])

ridge_pipeline.fit(X_train_raw, y_train)
ridge_pred = ridge_pipeline.predict(X_valid_raw)

ridge_rmse = rmse(y_valid, ridge_pred)
ridge_ic, ridge_pval = ic(y_valid, ridge_pred)

print("Ridge Regression Results:")
print(f"  RMSE         : {ridge_rmse:.2f}   (naive baseline: {baseline:.2f})")
print(f"  IC (Spearman): {ridge_ic:.4f}  (p-value: {ridge_pval:.4f})")
print()
if ridge_rmse > baseline:
    print("⚠️  RIDGE IS WORSE THAN THE NAIVE BASELINE ON RMSE!")
    print("   This is the expected result. Why?")
    print("   Ridge tries to predict absolute values — but our outliers are so extreme")
    print("   that the model's errors on them dominate the RMSE completely.")
    print()
    print("   BUT notice the IC: is it positive? That means Ridge ranks stocks correctly")
    print("   even if it can't predict the absolute magnitude.")
else:
    print("✓ Ridge beats the naive baseline on RMSE!")

In [ ]:
# 📊 Exercise 2 — Inspect Ridge coefficients (what did the model learn?)

imputer = ridge_pipeline.named_steps["imputer"]
scaler  = ridge_pipeline.named_steps["scaler"]
ridge   = ridge_pipeline.named_steps["ridge"]

feature_names = X_train_raw.columns.tolist()
coefs = pd.Series(ridge.coef_, index=feature_names)

# ── Plot top 20 largest absolute coefficients ──────────────────────────────────
top_pos = coefs[~coefs.index.str.endswith("_miss")].nlargest(10)
top_neg = coefs[~coefs.index.str.endswith("_miss")].nsmallest(10)
top_all = pd.concat([top_pos, top_neg]).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#EF4444" if v < 0 else "#22C55E" for v in top_all.values]
bars = ax.barh(range(len(top_all)), top_all.values, color=colors, edgecolor="white")
ax.set_yticks(range(len(top_all)))
ax.set_yticklabels(top_all.index, fontsize=9)
ax.axvline(0, color="black", linewidth=1)
ax.set_title("Ridge Regression Coefficients\n(green = predicts higher return, red = predicts lower return)", fontsize=11)
ax.set_xlabel("Coefficient (after scaling)")
plt.tight_layout()
plt.show()

print("🧠 Sanity checks:")
if "pe_ttm" in coefs.index:
    sign = "✓" if coefs["pe_ttm"] < 0 else "✗"
    print(f"  {sign} pe_ttm coefficient: {coefs['pe_ttm']:.4f}  "
          f"(expect negative — cheap stocks outperform)")
if "roe" in coefs.index:
    sign = "✓" if coefs["roe"] > 0 else "✗"
    print(f"  {sign} roe coefficient:    {coefs['roe']:.4f}  "
          f"(expect positive — profitable firms outperform)")
if "debt_to_equity" in coefs.index:
    sign = "✓" if coefs["debt_to_equity"] < 0 else "✗"
    print(f"  {sign} debt_to_equity:     {coefs['debt_to_equity']:.4f}  "
          f"(expect negative — more debt = more risk)")

print()
print("🔧 YOUR TURN:")
print("  Which coefficient surprised you most? Does it make economic sense?")
print("  Modify alpha in Ridge(alpha=???) and re-run — how do coefficients change?")

---
## 🔧 Exercise 2b — Fix the Ridge Model: Target Winsorisation *(10 min)*

The naive baseline beats Ridge because of extreme outliers in `return_pct`.

**The fix:** clip `return_pct` to the 1st–99th percentile of the *training set* before fitting.

This is called **target winsorisation** — a standard technique in quant finance.

⚠️ **Critical:** compute clip bounds on TRAIN FOLD ONLY, then clip train before fitting.
Do NOT clip the validation targets — we want to measure real performance on the validation set.

In [ ]:
# 🔧 Exercise 2b — Target winsorisation

# Step 1: Compute clip bounds on training data only
clip_lo = train_fold[TARGET].quantile(0.01)
clip_hi = train_fold[TARGET].quantile(0.99)

print(f"Clip bounds (from training data only):")
print(f"  1st percentile: {clip_lo:.2f}%")
print(f"  99th percentile: {clip_hi:.2f}%")
print()

# Step 2: Clip the training target
y_train_clipped = np.clip(y_train, clip_lo, clip_hi)

print(f"Before clipping: mean={y_train.mean():.2f}%  std={y_train.std():.2f}%  max={y_train.max():.2f}%")
print(f"After  clipping: mean={y_train_clipped.mean():.2f}%  std={y_train_clipped.std():.2f}%  max={y_train_clipped.max():.2f}%")
print()

# Step 3: Refit Ridge on clipped target (features unchanged)
ridge_clip = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge(alpha=1.0)),
])
ridge_clip.fit(X_train_raw, y_train_clipped)
ridge_clip_pred = ridge_clip.predict(X_valid_raw)

ridge_clip_rmse = rmse(y_valid, ridge_clip_pred)
ridge_clip_ic, _ = ic(y_valid, ridge_clip_pred)

print("Comparison:")
print(f"                 RMSE      IC")
print(f"  Naive baseline {baseline:.2f}    —")
print(f"  Ridge (raw y)  {ridge_rmse:.2f}   {ridge_ic:.4f}")
print(f"  Ridge (clip y) {ridge_clip_rmse:.2f}   {ridge_clip_ic:.4f}  ← does clipping help?")
print()
if ridge_clip_rmse < baseline:
    print("✓ Clipping the target makes Ridge beat the naive baseline!")
elif ridge_clip_rmse < ridge_rmse:
    print("✓ Clipping improved Ridge — still working on beating the naive baseline")
else:
    print("Hmm — clipping didn't help much here. Consider trying alpha values or more features.")

---
## Section 6 — Evaluation Metrics *(5 min)*

### Slide 18: Two metrics, two questions

| Metric | Question | Direction | Kaggle? |
|--------|----------|-----------|---------|
| RMSE   | How accurate are my absolute predictions? | Lower = better | ✓ Official |
| IC     | Do I rank stocks in the right order? | Higher = better | Secondary |

**Why IC matters for quant finance:**

Even if your model's absolute predictions are wrong (RMSE is high), if it correctly
*ranks* stocks — identifying which will outperform — you can build a profitable portfolio.

**IC benchmarks (from academic literature):**
- IC < 0: model is anti-predictive (sell what it says buy)
- IC 0.00–0.02: noise / barely detectable signal
- IC 0.02–0.05: weak but usable signal (many professional models here)
- IC 0.05–0.10: good signal — commercially valuable
- IC > 0.10: excellent — check for data leakage first!

**ICIR (IC Information Ratio):** IC / std(IC) across time periods. Measures *consistency*
of the signal — a stable IC of 0.03 is better than a noisy IC that averages 0.05.

In [ ]:
# 📊 Exercise 3 — Full evaluation dashboard

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Model Evaluation: Ridge (clipped target)", fontsize=12, fontweight="bold")

preds = ridge_clip_pred
y_true = y_valid

# ── Panel 1: Predicted vs Actual (clipped view) ────────────────────────────────
ax = axes[0]
clip_v = 200  # clip for display
ax.scatter(preds.clip(-clip_v, clip_v),
           y_true.clip(-clip_v, clip_v),
           alpha=0.15, s=8, color="#6366F1")
ax.plot([-clip_v, clip_v], [-clip_v, clip_v], "r--", linewidth=1.5, label="Perfect")
ax.set_xlabel("Predicted return_pct")
ax.set_ylabel("Actual return_pct")
ax.set_title(f"Predicted vs Actual\nRMSE={ridge_clip_rmse:.1f}  IC={ridge_clip_ic:.4f}")
ax.legend()

# ── Panel 2: Rank decile returns ───────────────────────────────────────────────
ax = axes[1]
df_eval = pd.DataFrame({"pred": preds, "actual": y_true})
df_eval["decile"] = pd.qcut(df_eval["pred"], q=10, labels=False) + 1
decile_ret = df_eval.groupby("decile")["actual"].mean()

bars = ax.bar(decile_ret.index, decile_ret.values,
              color=["#EF4444" if v < 0 else "#22C55E" for v in decile_ret.values],
              edgecolor="white")
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Predicted return decile (1=lowest, 10=highest)")
ax.set_ylabel("Mean actual return_pct (%)")
ax.set_title("Decile bar chart\n(best model = monotone increasing)")
ax.set_xticks(range(1, 11))

# ── Panel 3: Cumulative IC stability ──────────────────────────────────────────
ax = axes[2]
# Compute IC per year in the validation set
years = valid_fold["start_year"].unique()
df_yr_eval = pd.DataFrame({
    "pred": preds,
    "actual": y_true,
    "year": valid_fold["start_year"].values
})
ic_by_year = df_yr_eval.groupby("year").apply(
    lambda g: spearmanr(g["pred"], g["actual"])[0]
).reset_index()
ic_by_year.columns = ["year", "IC"]

colors_bar = ["#22C55E" if v > 0 else "#EF4444" for v in ic_by_year["IC"]]
ax.bar(ic_by_year["year"], ic_by_year["IC"], color=colors_bar, edgecolor="white")
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Year")
ax.set_ylabel("IC (Spearman)")
ax.set_title("IC by year\n(stability matters more than raw IC)")

plt.tight_layout()
plt.show()

print(f"\n📊 Summary:")
print(f"   RMSE        : {ridge_clip_rmse:.2f}   (naive: {baseline:.2f})")
print(f"   IC          : {ridge_clip_ic:.4f}")
print(f"   Decile spread (D10 - D1): {decile_ret.iloc[-1] - decile_ret.iloc[0]:.2f}%")
print()
print("🧠 The decile chart is key: if decile 10 has the highest average return,")
print("   you could build a portfolio of top-decile stocks and outperform.")

---
## Section 7 — LightGBM: Beyond Linearity *(~10 min)*

### Slide 19: Why gradient boosted trees win on tabular financial data

**Gradient boosting** trains a sequence of shallow decision trees.
Each tree learns to correct the residual errors of the previous ensemble.

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

where $h_m$ is the $m$-th tree fitted on negative gradients of the loss.

**Why trees beat Ridge for this problem:**
1. **Non-linearity**: "low P/E AND high ROE AND low debt" is much better than each alone
2. **Interactions**: Ridge must be told about interactions explicitly; trees find them automatically
3. **Missing values**: LightGBM handles NaN natively — we can pass the transformed DataFrame directly
4. **Robustness**: less sensitive to remaining outliers in the target

**Key hyperparameters:**
- `n_estimators`: number of trees — more = more capacity, but risk overfitting
- `num_leaves`: max leaves per tree (32 = shallow, 256 = deep)
- `learning_rate`: shrinkage — smaller = slower but better generalisation
- `min_child_samples`: minimum samples per leaf — regularisation

In [ ]:
# ── LightGBM with early stopping ───────────────────────────────────────────────

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    print("LightGBM not available — run: pip install lightgbm --break-system-packages")
    HAS_LGB = False

if HAS_LGB:
    # LightGBM handles NaNs natively; keep the missingness flags as extra signals.
    X_tr_lgb = X_train_raw
    X_va_lgb = X_valid_raw
    X_te_lgb = X_test_raw

    # Use clipped target for training
    lgb_model = lgb.LGBMRegressor(
        n_estimators      = 500,
        learning_rate     = 0.05,
        num_leaves        = 31,
        max_depth         = 4,
        min_child_samples = 30,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        reg_alpha         = 0.1,
        reg_lambda        = 0.1,
        random_state      = 42,
        verbosity         = -1,
    )

    lgb_model.fit(
        X_tr_lgb, y_train_clipped,
        eval_set=[(X_va_lgb, y_valid)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
    )

    lgb_pred = lgb_model.predict(X_va_lgb)
    lgb_rmse = rmse(y_valid, lgb_pred)
    lgb_ic, _  = ic(y_valid, lgb_pred)

    print("LightGBM Results:")
    print(f"  Best iteration : {lgb_model.best_iteration_}")
    print()
    print(f"                 RMSE      IC")
    print(f"  Naive baseline {baseline:.2f}    —")
    print(f"  Ridge (clip y) {ridge_clip_rmse:.2f}   {ridge_clip_ic:.4f}")
    print(f"  LightGBM       {lgb_rmse:.2f}   {lgb_ic:.4f}  ← improvement?")

In [ ]:
# 📊 Exercise 4 — Feature importances

if HAS_LGB:
    feat_names = X_train_raw.columns.tolist()
    imp = (pd.Series(lgb_model.feature_importances_, index=feat_names)
           .sort_values(ascending=False))

    # Separate ratio/scale from miss flags
    imp_main = imp[~imp.index.str.endswith("_miss")].head(15)
    imp_miss = imp[imp.index.str.endswith("_miss")].head(10)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("LightGBM Feature Importances", fontsize=12, fontweight="bold")

    # ── Main features ──────────────────────────────────────────────────────────
    ax = axes[0]
    ax.barh(range(len(imp_main)), imp_main.values, color="#6366F1", edgecolor="white")
    ax.set_yticks(range(len(imp_main)))
    ax.set_yticklabels(imp_main.index, fontsize=9)
    ax.invert_yaxis()
    ax.set_title("Top 15 main features")
    ax.set_xlabel("Importance (split count)")

    # ── Missingness flags ──────────────────────────────────────────────────────
    ax = axes[1]
    ax.barh(range(len(imp_miss)), imp_miss.values, color="#F97316", edgecolor="white")
    ax.set_yticks(range(len(imp_miss)))
    ax.set_yticklabels(imp_miss.index, fontsize=9)
    ax.invert_yaxis()
    ax.set_title("Top 10 missingness flags\n(absence as signal)")
    ax.set_xlabel("Importance (split count)")

    plt.tight_layout()
    plt.show()

    print("🧠 Questions to discuss:")
    print("  1. Which feature is most important? Does that make economic sense?")
    print("  2. Are missingness flags in the top 20? Why or why not?")
    print("  3. Does LightGBM improve IC over Ridge? What does that tell us?")
    print()
    print(f"🔧 YOUR TURN:")
    print(f"  Try changing num_leaves from 31 to 127 — does RMSE/IC improve?")
    print(f"  Try adding 'start_year' as a feature — what happens to feature importances?")
    print(f"  (Hint: if start_year ranks #1, you might have a regime memorisation problem)")

---
## Section 8 — Pitfalls: Expanding Window Cross-Validation *(~10 min)*

### Slide 20: Why single-year validation is not enough

A single 2022 validation set is one year — one market regime. Your model could:
- Work perfectly in 2022 (a bear market year) but fail in bull markets
- Overfit hyperparameters to 2022 specifically

**Expanding window CV** mimics real deployment:
```
Fold 1: Train 2019       → Validate 2020
Fold 2: Train 2019-2020  → Validate 2021
Fold 3: Train 2019-2021  → Validate 2022
```

This gives you IC and RMSE across multiple regimes — a much more honest estimate.
Do not panic if one fold is negative; instability is information, not a notebook failure.

⚠️ **Never use sklearn KFold** on time-series data — it randomly shuffles observations and
creates look-ahead bias where your model "knows" 2022 when fitting on 2019.

In [ ]:
# 📊 Expanding window cross-validation

print("Expanding Window Validation")
print("=" * 55)

all_years  = sorted(train["start_year"].unique())
folds      = []
cv_results = []

for val_yr in all_years[1:]:  # start from 2nd year (need at least 1 train year)
    train_yrs_fold = [y for y in all_years if y < val_yr]
    if not train_yrs_fold:
        continue

    tr = train[train["start_year"].isin(train_yrs_fold)]
    va = train[train["start_year"] == val_yr]

    if len(va) < 50:
        continue  # skip tiny folds

    # Features
    X_tr_f, cs_f, ss_f = build_features(tr, fit=True)
    X_va_f, _, _        = build_features(va, clip_stats=cs_f, sector_stats=ss_f, fit=False)
    y_tr_f = tr[TARGET].values
    y_va_f = va[TARGET].values

    # Clip target on this fold
    lo_f = np.percentile(y_tr_f, 1)
    hi_f = np.percentile(y_tr_f, 99)
    y_tr_clip_f = np.clip(y_tr_f, lo_f, hi_f)

    # Ridge on clipped target
    pipe_f = Pipeline([
        ("imp",    SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge",  Ridge(alpha=1.0)),
    ])
    pipe_f.fit(X_tr_f, y_tr_clip_f)
    pred_f = pipe_f.predict(X_va_f)

    fold_ic, _  = ic(y_va_f, pred_f)
    fold_rmse   = rmse(y_va_f, pred_f)
    fold_naive  = rmse(y_va_f, np.full(len(y_va_f), y_tr_f.mean()))

    cv_results.append({
        "val_year":   val_yr,
        "train_yrs":  len(train_yrs_fold),
        "n_train":    len(tr),
        "n_valid":    len(va),
        "IC":         fold_ic,
        "RMSE":       fold_rmse,
        "Naive_RMSE": fold_naive,
    })
    print(f"  Train {train_yrs_fold} → Val {val_yr}: "
          f"IC={fold_ic:+.4f}  RMSE={fold_rmse:.1f}  (naive: {fold_naive:.1f})")

cv_df = pd.DataFrame(cv_results)

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("Expanding Window CV — Ridge with clipped target", fontsize=11, fontweight="bold")

ax = axes[0]
ax.bar(cv_df["val_year"], cv_df["IC"],
       color=["#22C55E" if v > 0 else "#EF4444" for v in cv_df["IC"]],
       edgecolor="white")
ax.axhline(0, color="black", linewidth=1)
ax.axhline(cv_df["IC"].mean(), color="blue", linewidth=1.5, linestyle="--",
           label=f"Mean IC = {cv_df['IC'].mean():.4f}")
ax.set_xlabel("Validation year"); ax.set_ylabel("IC")
ax.set_title("IC across validation years"); ax.legend()

ax = axes[1]
ax.plot(cv_df["val_year"], cv_df["RMSE"], "o-", color="#6366F1", label="Ridge RMSE")
ax.plot(cv_df["val_year"], cv_df["Naive_RMSE"], "s--", color="#F97316", label="Naive RMSE")
ax.set_xlabel("Validation year"); ax.set_ylabel("RMSE")
ax.set_title("RMSE: Ridge vs Naive baseline"); ax.legend()

plt.tight_layout()
plt.show()

print(f"\nExpanding CV Summary:")
print(f"  Mean IC    : {cv_df['IC'].mean():.4f}")
print(f"  IC Std     : {cv_df['IC'].std():.4f}")
print(f"  ICIR       : {cv_df['IC'].mean()/max(cv_df['IC'].std(),1e-6):.2f}  (IC/std — stability metric)")
print(f"  Beats naive: {(cv_df['RMSE'] < cv_df['Naive_RMSE']).sum()}/{len(cv_df)} folds")

---
## Final Step — Generate Kaggle Submission *(5 min)*

Refit the best model on **all** training data (2019–2022), then predict 2024 test set.

**Why refit on all training data?**
- More data generally = better model
- We've already validated our approach — no more hyperparameter tuning allowed now
- The Kaggle submission file must have columns: `id`, `return_pct`

In [ ]:
# ── Refit on full training data ────────────────────────────────────────────────

print("Refitting on all training data (2019–2022)…")

X_full, cs_full, ss_full = build_features(train, fit=True)
y_full = train[TARGET].values

# Clip full training target
clip_lo_full = np.percentile(y_full, 1)
clip_hi_full = np.percentile(y_full, 99)
y_full_clipped = np.clip(y_full, clip_lo_full, clip_hi_full)

# Ridge on full data
final_ridge = Pipeline([
    ("imp",    SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge",  Ridge(alpha=1.0)),
])
final_ridge.fit(X_full, y_full_clipped)

# Predict test
X_test_full, _, _ = build_features(test, clip_stats=cs_full, sector_stats=ss_full, fit=False)
test_pred = final_ridge.predict(X_test_full)

# ── Submission file ───────────────────────────────────────────────────────────
sub = pd.DataFrame({
    ID_COL:  test[ID_COL],
    TARGET:  test_pred,
})
sub_path = SUB_DIR / "ridge_workshop_submission.csv"
sub.to_csv(sub_path, index=False)

print(f"✓  Saved: {sub_path}")
print(f"   Shape : {sub.shape}")
print(f"   Prediction range: [{test_pred.min():.1f}%, {test_pred.max():.1f}%]")
print()
print("Upload this CSV to the Kaggle competition and check your score!")
print("Then come back and try:")
print("  1. Replace Ridge with LightGBM in this final step")
print("  2. Average (blend) Ridge + LightGBM predictions")
print("  3. Add more features from 03_feature_engineering_strategy.ipynb")

In [ ]:
# 📊 BONUS — LightGBM submission (if you have time)

if HAS_LGB:
    print("Training LightGBM on full data for submission…")

    X_full_lgb = X_full
    X_test_lgb = X_test_full

    lgb_final = lgb.LGBMRegressor(
        n_estimators  = lgb_model.best_iteration_ or 300,
        learning_rate = 0.05,
        num_leaves    = 31,
        max_depth     = 4,
        min_child_samples = 30,
        subsample     = 0.8,
        colsample_bytree = 0.8,
        reg_alpha     = 0.1,
        reg_lambda    = 0.1,
        random_state  = 42,
        verbosity     = -1,
    )
    lgb_final.fit(X_full_lgb, y_full_clipped)
    test_pred_lgb = lgb_final.predict(X_test_lgb)

    # Blend: 50% Ridge + 50% LightGBM (reduces variance)
    blend_pred = 0.5 * test_pred + 0.5 * test_pred_lgb

    lgb_sub   = pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred_lgb})
    blend_sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET: blend_pred})

    lgb_sub.to_csv(SUB_DIR / "lgbm_workshop_submission.csv",  index=False)
    blend_sub.to_csv(SUB_DIR / "blend_workshop_submission.csv", index=False)

    print(f"✓  LightGBM submission: {SUB_DIR}/lgbm_workshop_submission.csv")
    print(f"✓  Blend (50/50)  sub : {SUB_DIR}/blend_workshop_submission.csv")
    print()
    print("📊 Compare Ridge vs LightGBM predictions:")
    print(f"   Ridge range   : [{test_pred.min():.1f}%, {test_pred.max():.1f}%]")
    print(f"   LightGBM range: [{test_pred_lgb.min():.1f}%, {test_pred_lgb.max():.1f}%]")
    print(f"   Blend range   : [{blend_pred.min():.1f}%, {blend_pred.max():.1f}%]")

---
## What to Try Next

Work through these in order — each builds on the previous:

**Immediate improvements:**
1. **Better target**: try rank-transforming `y`: `y_ranked = y.rank(pct=True)` — this makes the target uniform [0,1] and often improves IC
2. **XGBoost**: try `xgb.XGBRegressor` — similar to LightGBM but often complementary for blending
3. **Feature interactions**: manually add `pe_ttm * roe`, `gross_margin * revenue_growth_yoy`

**Advanced (see `03_feature_engineering_strategy.ipynb`):**
4. **7-step feature pipeline**: the production-grade version of what we built today
5. **Sector-relative target**: predict `return_pct - sector_median_return` instead of raw return
6. **Purged cross-validation**: add embargo period between train and validation to prevent leakage

**Leaderboard strategy:**
7. Submit your best single model first
8. Blend Ridge + LightGBM + XGBoost — averaging reduces variance
9. Look at top Kaggle kernels for this competition for ideas

---
*AIC Quant Meeting · Research Track · June 2026*

*Questions? → open a GitHub issue on the competition repo*

*Sources: Fama & French (1993), McLean & Pontiff (2016 — "Does academic research destroy stock return predictability?"), Lopez de Prado (2018 — "Advances in Financial Machine Learning")*